### KRX/FDR Firm Characteristics Data Collection

In [ ]:
import FinanceDataReader as fdr
import pandas as pd
import math

# ── 1. Collect KRX-listed company information ─────────────────────

kospi_info = fdr.StockListing("KOSPI")
kosdaq_info = fdr.StockListing("KOSDAQ")

# Convert column names to lowercase first
kospi_info.columns = [c.lower() for c in kospi_info.columns]
kosdaq_info.columns = [c.lower() for c in kosdaq_info.columns]

# Remove the existing market column if it exists
kospi_info = kospi_info.drop(columns=["market"], errors="ignore")
kosdaq_info = kosdaq_info.drop(columns=["market"], errors="ignore")

# Create the market classification manually
kospi_info["market"] = "KOSPI"
kosdaq_info["market"] = "KOSDAQ"

listing_df = pd.concat([kospi_info, kosdaq_info], ignore_index=True)

# Prevent duplicate columns
listing_df = listing_df.loc[:, ~listing_df.columns.duplicated()]

listing_df["code"] = listing_df["code"].astype(str).str.zfill(6)

print("KRX 컬럼 목록:")
print(listing_df.columns.tolist())


# ── 2. Load the list of companies included in the analysis ─────────

disclosure_df = pd.read_csv(
    "dart_earnings_disclosure_list_2019_2026.csv",
    dtype=str
)

corps = (
    disclosure_df[["corp_code", "stock_code", "corp_name", "market"]]
    .drop_duplicates(subset=["corp_code"])
    .reset_index(drop=True)
)

corps["corp_code"] = corps["corp_code"].astype(str).str.zfill(8)
corps["stock_code"] = corps["stock_code"].astype(str).str.zfill(6)

print(f"\n분석 대상 기업 수: {len(corps)}")


# ── 3. Select FDR/KRX firm characteristic columns ──────────────────

wanted_cols = [
    "code",
    "name",
    "market",
    "sector",
    "industry",
    "foreignratio",
    "marcap"
]

available_cols = [c for c in wanted_cols if c in listing_df.columns]

print("\n사용 가능한 KRX/FDR 컬럼:")
print(available_cols)

krx_controls = listing_df[available_cols].copy()


# ── 4. Merge with the company list ─────────────────────────────────

company_controls = corps.merge(
    krx_controls,
    left_on="stock_code",
    right_on="code",
    how="left",
    suffixes=("", "_krx")
)


# ── 5. Validation: Check for market classification mismatches ──────

if "market_krx" in company_controls.columns:
    mismatch = company_controls[
        company_controls["market"].astype(str) != company_controls["market_krx"].astype(str)
    ]

    print(f"\n시장구분 불일치 기업 수: {len(mismatch)}")

    if len(mismatch) > 0:
        print(
            mismatch[
                ["corp_code", "stock_code", "corp_name", "market", "market_krx"]
            ].head()
        )


# ── 6. Validation: Check for unmatched KRX/FDR companies ───────────

unmatched = company_controls[company_controls["code"].isna()]

print(f"\nKRX/FDR 매칭 실패 기업 수: {len(unmatched)}")

if len(unmatched) > 0:
    print(
        unmatched[
            ["corp_code", "stock_code", "corp_name", "market"]
        ].head()
    )


# ── 7. Prepare variables for analysis ──────────────────────────────

if "marcap" in company_controls.columns:
    company_controls["marcap"] = pd.to_numeric(
        company_controls["marcap"],
        errors="coerce"
    )

    company_controls["log_marcap"] = company_controls["marcap"].apply(
        lambda x: pd.NA if pd.isna(x) or x <= 0 else math.log(x)
    )

if "foreignratio" in company_controls.columns:
    company_controls["foreignratio"] = pd.to_numeric(
        company_controls["foreignratio"],
        errors="coerce"
    )


# ── 8. Save and perform final checks ───────────────────────────────

output_file = "company_krx_fdr_controls_2019_2026.csv"

company_controls.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료: {output_file}")
print(company_controls.shape)
print(company_controls.head())

print("\n결측치 비율:")
print(company_controls.isna().mean().sort_values(ascending=False))

KRX 컬럼 목록:
['code', 'isu_cd', 'name', 'dept', 'close', 'changecode', 'changes', 'chagesratio', 'open', 'high', 'low', 'volume', 'amount', 'marcap', 'stocks', 'marketid', 'market']

분석 대상 기업 수: 947

사용 가능한 KRX/FDR 컬럼:
['code', 'name', 'market', 'marcap']

시장구분 불일치 기업 수: 0

KRX/FDR 매칭 실패 기업 수: 0

저장 완료: company_krx_fdr_controls_2019_2026.csv
(947, 9)
  corp_code stock_code corp_name  market    code      name market_krx  \
0  01267170     285130     SK케미칼   KOSPI  285130     SK케미칼      KOSPI   
1  00684714     103140        풍산   KOSPI  103140        풍산      KOSPI   
2  00657002     200710  에이디테크놀로지  KOSDAQ  200710  에이디테크놀로지     KOSDAQ   
3  00414850     094280    효성 ITX   KOSPI  094280     효성ITX      KOSPI   
4  01316236     298000      효성화학   KOSPI  298000      효성화학      KOSPI   

          marcap  log_marcap  
0   842449904200   27.459580  
1  2457729180600   28.530259  
2   603771013950   27.126461  
3   146789140000   25.712263  
4   147501447900   25.717104  

결측치 비율:
corp_code     0

### DART Annual Report Financial Statement Data Collection

In [ ]:
import time
import requests
import pandas as pd
import math
from tqdm.auto import tqdm

# =========================================
# 0. Configuration
# =========================================

DART_API_KEY = os.getenv("DART_API_KEY")

INPUT_FILE = "dart_earnings_disclosure_list_2019_2026.csv"
OUTPUT_FILE = "dart_financial_controls_2019_2025.csv"

START_YEAR = 2019
END_YEAR = 2025

REPORT_CODE = "11011"  # Annual report

# Account name mapping
TARGET_ACCOUNTS = {
    "자산총계": "total_assets",
    "부채총계": "total_liabilities",
    "자본총계": "total_equity",

    "매출액": "revenue",
    "영업수익": "revenue",

    "영업이익": "operating_income",

    "당기순이익": "net_income",
    "분기순이익": "net_income",
    "반기순이익": "net_income",
}


# =========================================
# 1. Load the list of companies included in the analysis
# =========================================

disclosure_df = pd.read_csv(INPUT_FILE, dtype=str)

corps = (
    disclosure_df[["corp_code", "stock_code", "corp_name", "market"]]
    .drop_duplicates(subset=["corp_code"])
    .reset_index(drop=True)
)

corps["corp_code"] = corps["corp_code"].astype(str).str.zfill(8)
corps["stock_code"] = corps["stock_code"].astype(str).str.zfill(6)

print(f"수집 대상 기업 수: {len(corps)}")


# =========================================
# 2. DART financial statement API function
# =========================================

def fetch_financial_statement(corp_code, year):

    url = "https://opendart.fss.or.kr/api/fnlttSinglAcnt.json"

    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code,
        "bsns_year": str(year),
        "reprt_code": REPORT_CODE
    }

    try:
        res = requests.get(url, params=params, timeout=30)
        data = res.json()

        status = data.get("status")
        message = data.get("message")

        # Handle non-successful responses
        if status != "000":
            return {
                "corp_code": corp_code,
                "year": year,
                "status": status,
                "message": message
            }

        rows = data.get("list", [])

        result = {
            "corp_code": corp_code,
            "year": year,
            "status": status,
            "message": message
        }

        df = pd.DataFrame(rows)

        # Prioritize consolidated financial statements
        if "fs_div" in df.columns:

            if (df["fs_div"] == "CFS").any():
                df = df[df["fs_div"] == "CFS"]

            elif (df["fs_div"] == "OFS").any():
                df = df[df["fs_div"] == "OFS"]

        # ---------------------------------
        # Improved matching using partial account name matches
        # ---------------------------------

        for _, row in df.iterrows():

            account_name = str(row.get("account_nm", "")).strip()
            amount = row.get("thstrm_amount")

            if pd.isna(amount):
                continue

            amount = str(amount).replace(",", "").strip()

            numeric_amount = pd.to_numeric(
                amount,
                errors="coerce"
            )

            # Partial-match account names
            for key, var_name in TARGET_ACCOUNTS.items():

                if key in account_name:

                    # Do not overwrite an existing value
                    if var_name not in result:
                        result[var_name] = numeric_amount

        return result

    except Exception as e:

        return {
            "corp_code": corp_code,
            "year": year,
            "status": "ERROR",
            "message": str(e)
        }


# =========================================
# 3. Collect financial data by company and year
# =========================================

financial_rows = []

years = range(START_YEAR, END_YEAR + 1)

for _, corp in tqdm(corps.iterrows(), total=len(corps)):

    corp_code = corp["corp_code"]

    for year in years:

        item = fetch_financial_statement(
            corp_code,
            year
        )

        financial_rows.append(item)

        # Prevent exceeding API rate limits
        time.sleep(0.5)

financial_df = pd.DataFrame(financial_rows)


# =========================================
# 4. Merge company information
# =========================================

final_df = financial_df.merge(
    corps,
    on="corp_code",
    how="left"
)


# =========================================
# 5. Create derived variables for analysis
# =========================================

numeric_cols = [
    "total_assets",
    "total_liabilities",
    "total_equity",
    "revenue",
    "operating_income",
    "net_income"
]

for col in numeric_cols:

    if col in final_df.columns:

        final_df[col] = pd.to_numeric(
            final_df[col],
            errors="coerce"
        )


# -----------------------------------------
# Debt ratio
# -----------------------------------------

if {
    "total_liabilities",
    "total_equity"
}.issubset(final_df.columns):

    final_df["debt_ratio"] = (
        final_df["total_liabilities"]
        / final_df["total_equity"]
    )


# -----------------------------------------
# ROA
# -----------------------------------------

if {
    "net_income",
    "total_assets"
}.issubset(final_df.columns):

    final_df["roa"] = (
        final_df["net_income"]
        / final_df["total_assets"]
    )


# -----------------------------------------
# Operating margin
# -----------------------------------------

if {
    "operating_income",
    "revenue"
}.issubset(final_df.columns):

    final_df["operating_margin"] = (
        final_df["operating_income"]
        / final_df["revenue"]
    )


# -----------------------------------------
# Log of total assets
# -----------------------------------------

if "total_assets" in final_df.columns:

    final_df["log_assets"] = final_df["total_assets"].apply(
        lambda x:
        pd.NA
        if pd.isna(x) or x <= 0
        else math.log(x)
    )


# =========================================
# 6. Save and validate the dataset
# =========================================

final_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n저장 완료: {OUTPUT_FILE}")

print("\n데이터 크기:")
print(final_df.shape)

print("\n상위 데이터:")
print(final_df.head())

print("\n결측치 비율:")
print(
    final_df.isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nDART 응답 상태:")
print(
    final_df["status"]
    .value_counts(dropna=False)
)

수집 대상 기업 수: 947


  0%|          | 0/947 [00:00<?, ?it/s]


저장 완료: dart_financial_controls_2019_2025.csv

데이터 크기:
(6629, 17)

상위 데이터:
  corp_code  year status message  total_assets  total_liabilities  \
0  01267170  2019    000      정상  2.131709e+12       1.413003e+12   
1  01267170  2020    000      정상  2.119967e+12       1.161034e+12   
2  01267170  2021    000      정상  3.955632e+12       1.388895e+12   
3  01267170  2022    000      정상  3.945683e+12       1.236777e+12   
4  01267170  2023    000      정상  4.158775e+12       1.483001e+12   

   total_equity       revenue  operating_income    net_income stock_code  \
0  7.187063e+11  1.427186e+12      8.033782e+10  5.021554e+09     285130   
1  9.589328e+11  1.214709e+12      1.062830e+11  2.551612e+11     285130   
2  2.566737e+12  2.089632e+12      5.551859e+11  2.687433e+11     285130   
3  2.708906e+12  1.829191e+12      2.304807e+11  2.314759e+11     285130   
4  2.675774e+12  1.748778e+12      8.330249e+10  4.783789e+10     285130   

  corp_name market  debt_ratio       roa  operating_m

## 4. Collection of Firm Financial Data and Control Variables

### 4-1. Purpose of Data Collection

In the analysis of disclosure timing and stock market reactions, firm characteristics such as firm size and financial condition may serve as important control variables.

For example, larger firms may have more developed disclosure systems and investor communication capabilities, while financially weaker firms may exhibit different patterns of bad-news disclosure or market reactions.

Therefore, this study constructed two types of control variables to account for firm-specific characteristics:

```text
1. KRX/FDR-based firm characteristic variables
2. DART annual report-based financial control variables
```

---

## 4-2. Construction of KRX/FDR-Based Firm Characteristic Variables

### Purpose of Data Collection

Firm-level market characteristics were collected using data from KRX and FinanceDataReader (FDR).

This process provided the following firm-level characteristics:

- Market classification (KOSPI/KOSDAQ)
- Market capitalization
- Log market capitalization

These variables were used to control for the effects of firm size and market characteristics on disclosure timing and stock market reactions.

---

### Data Collection Method

First, lists of KOSPI- and KOSDAQ-listed firms were collected using the `FinanceDataReader.StockListing()` function.

```python
fdr.StockListing("KOSPI")
fdr.StockListing("KOSDAQ")
```

The collected listing data contained various firm-level information, including stock codes, company names, market classifications, and market capitalization.

The listing data were then merged with the sample of firms in the earnings disclosure dataset (`dart_earnings_disclosure_list_2019_2026.csv`) using the stock code (`stock_code`). This restricted the dataset to firms with earnings disclosures included in the analysis.

For use in subsequent analyses, market capitalization (`marcap`) was converted to a numeric variable, and log market capitalization (`log_marcap`) was constructed.

```python
company_controls["log_marcap"] = company_controls["marcap"].apply(
    lambda x: math.log(x)
)
```

---

### Constructed Variables

The following firm characteristic variables were constructed:

| Variable | Description |
|---|---|
| market | Market classification (KOSPI/KOSDAQ) |
| marcap | Market capitalization |
| log_marcap | Log market capitalization |
| foreignratio | Foreign ownership ratio |
| sector | Sector |
| industry | Industry classification |

However, the availability of specific variables may vary depending on the information provided by KRX/FDR at the time of data collection.

---

### Data Storage

The constructed firm characteristic dataset was saved as:

```text
company_krx_fdr_controls_2019_2026.csv
```

---

## 4-3. Construction of DART Annual Report-Based Financial Control Variables

### Purpose of Data Collection

To control for firms' financial conditions, additional financial statement data were collected from annual reports through the DART OpenAPI.

These data were used to construct financial control variables at the firm-year level.

---

### Data Collection Method

Financial statement data for each firm were collected using DART OpenAPI's `fnlttSinglAcnt.json` endpoint.

The sample firms were identified from the earnings disclosure dataset, and annual report financial data were retrieved for each firm (`corp_code`) for the period from 2019 to 2025.

```python
fetch_financial_statement(corp_code, year)
```

Consolidated financial statements (CFS) were prioritized when available. If consolidated financial statements were unavailable, separate financial statements (OFS) were used instead.

```python
if (df["fs_div"] == "CFS").any():
    df = df[df["fs_div"] == "CFS"]
```

To account for variations in account names (`account_nm`), major financial items were extracted using partial string matching.

For example, the following account names were mapped to common variables:

```text
매출액 / 영업수익 → revenue

당기순이익 / 분기순이익 / 반기순이익 → net_income
```

---

### Constructed Variables

The following financial control variables were constructed:

| Variable | Description |
|---|---|
| total_assets | Total assets |
| total_liabilities | Total liabilities |
| total_equity | Total equity |
| revenue | Revenue |
| operating_income | Operating income |
| net_income | Net income |
| debt_ratio | Debt ratio |
| roa | Return on Assets (ROA) |
| operating_margin | Operating margin |
| log_assets | Log of total assets |

---

### Derived Variables

Based on the collected financial information, the following financial ratios and transformed variables were additionally constructed.

#### Debt Ratio

```python
debt_ratio =
    total_liabilities / total_equity
```

#### Return on Assets (ROA)

```python
roa =
    net_income / total_assets
```

#### Operating Margin

```python
operating_margin =
    operating_income / revenue
```

#### Log of Total Assets

```python
log_assets =
    log(total_assets)
```

---

### Data Storage

The final financial control variable dataset was saved as:

```text
dart_financial_controls_2019_2025.csv
```

Financial control variables for 2026 were excluded because annual reports for the 2026 fiscal year were not yet available at the time of data collection.

---

## 4-4. Role in the Analysis

The firm characteristics and financial control variables constructed in this stage were used in subsequent regression analyses for the following purposes:

| Variable Group | Purpose |
|---|---|
| Market classification | Compare KOSPI and KOSDAQ firms |
| Market capitalization | Control for firm size |
| Log market capitalization | Control for firm size effects |
| ROA | Control for profitability |
| Debt ratio | Control for financial condition |
| Foreign ownership ratio | Control for investor ownership structure |
| Industry variables | Control for industry-level differences |

In particular, these control variables were included in the second-stage regression analysis of **Analysis ③** to estimate the effect of disclosure timing on CAR while controlling for observable firm characteristics.